# Elastic 2-D topography benchmark — September 2026

Progressive validation of OPT3 and OPT5 against SPECFEM2D. The flat homogeneous free surface is validated over a longer physical window before introducing Kirishima topography. FD3 is deliberately excluded from the principal figures.

In [ ]:
import Pkg
function find_flexopt_root(start=pwd())
    directory = abspath(start)
    while true
        isfile(joinpath(directory, "src", "flexOPT.jl")) && return directory
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    haskey(ENV, "FLEXOPT_ROOT") && return abspath(ENV["FLEXOPT_ROOT"])
    error("Cannot locate flexOPT; set ENV[\"FLEXOPT_ROOT\"]")
end
flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
using CairoMakie, JLD2, LinearAlgebra, Statistics
CairoMakie.activate!(type="png")
@show VERSION Threads.nthreads() Base.active_project()


## Stage 1 — long flat free-surface benchmark

The three nominal worker spacings 1000, 750 and 500 m become physical spacings 500, 375 and 250 m because the long profile activates the established factor-two refinement. Duration is 14 s and the half-domain is 90 km.

In [ ]:
runLongFlatBenchmark = false # expensive: set true explicitly
longBenchmarkCommand = `$(Base.julia_cmd()) --project=$(flexopt_root)
    --startup-file=no --threads=8
    $(joinpath(flexopt_root, "scripts", "run_flat_free_surface_convergence.jl"))
    --long-opt5 1000 750 500`
@show longBenchmarkCommand
runLongFlatBenchmark && run(longBenchmarkCommand)


In [ ]:
longDataDirectory = joinpath(flexopt_root, "data",
    "elastic2d_convergence", "flat_free_surface_long_opt5")
longFiles = isdir(longDataDirectory) ? sort(filter(
    file -> endswith(file, ".jld2"), readdir(longDataDirectory; join=true))) : String[]
longResults = sort([load(file, "result") for file in longFiles];
    by=result -> result.spacing_m, rev=true)
isempty(longResults) && @warn "No long OPT5 products yet; enable runLongFlatBenchmark once"
!isempty(longResults) && display([(dx=result.spacing_m,
    duration=result.duration_s, dt_OPT3=result.OPT3.dt,
    dt_OPT5=result.OPT5.dt, dt_SPECFEM=result.SPECFEM2D.dt)
    for result in longResults])


In [ ]:
function compact_trace(result, method, component, station)
    block = getproperty(result, method)
    values = getproperty(block, component)
    method === :SPECFEM2D ? values[station] :
        (time=values.time, values=values.values[:, station])
end
if !isempty(longResults)
    result = last(longResults)
    methods = (:OPT3, :OPT5, :SPECFEM2D)
    colors = Dict(:OPT3 => :darkorange, :OPT5 => :purple,
        :SPECFEM2D => :black)
    lateFigure = Figure(size=(1200, 225 * length(result.receivers)))
    for (station, receiver) in pairs(result.receivers)
        axis = Axis(lateFigure[station, 1]; xlabel="time (s)",
            ylabel="u_z (m)",
            title="late absolute phases, station $station: x=$(receiver.x/1e3), z=$(receiver.z/1e3) km",
            limits=(0.0, result.duration_s, nothing, nothing))
        for method in methods
            trace = compact_trace(result, method, :z, station)
            lines!(axis, trace.time, trace.values; color=colors[method],
                label=String(method))
        end
        station == 1 && axislegend(axis; position=:rt)
    end
    display(lateFigure)
end


In [ ]:
function sample_trace(trace, times)
    [begin
        upper = searchsortedfirst(trace.time, time)
        if upper <= 1
            trace.values[1]
        elseif upper > length(trace.time)
            trace.values[end]
        else
            fraction = (time-trace.time[upper-1]) /
                (trace.time[upper]-trace.time[upper-1])
            trace.values[upper-1] + fraction *
                (trace.values[upper]-trace.values[upper-1])
        end
    end for time in times]
end
if length(longResults) >= 2
    finest = last(longResults)
    convergenceFigure = Figure(size=(1100, 500))
    axis = Axis(convergenceFigure[1, 1]; xscale=log10, yscale=log10,
        xlabel="spacing (m)", ylabel="global relative RMS to finest",
        title="14 s flat-surface convergence")
    for (method, color) in ((:OPT3, :darkorange), (:OPT5, :purple),
            (:SPECFEM2D, :black))
        errors = Float64[]
        spacings = Float64[]
        for candidate in longResults[1:end-1]
            referenceValues, candidateValues = Float64[], Float64[]
            for station in eachindex(finest.receivers)
                reference = compact_trace(finest, method, :z, station)
                trial = compact_trace(candidate, method, :z, station)
                times = range(max(first(reference.time), first(trial.time)),
                    min(last(reference.time), last(trial.time)); length=1801)
                append!(referenceValues, sample_trace(reference, times))
                append!(candidateValues, sample_trace(trial, times))
            end
            push!(errors, norm(candidateValues-referenceValues) /
                norm(referenceValues))
            push!(spacings, candidate.spacing_m)
        end
        scatterlines!(axis, spacings, errors; color, label=String(method))
    end
    axislegend(axis)
    display(convergenceFigure)
end


## Stage 2 — Kirishima non-flat topography

This stage remains disabled until OPT3 and OPT5 complete the 14 s flat run without early stopping and show consistent convergence against SPECFEM2D (Stage 1 above). As of now only the dx=500 m point of that long run exists; dx=750 m and dx=1000 m are still missing, so this stage is being built ahead of that gate clearing, at the user's explicit request.

`topography-bootstrap`/`topography-model` build the Kirishima cross-section itself — real topography and the NIED heterogeneous velocity model — reusing only `constructLocalBox`/`getParamsAndTopo` from `SimuKirishima.ipynb`, at the same fine Δx=Δz=100 m used there (coarser grids do not converge; see Stage 1). "Air" here is not modelled as a medium: `material2D` only marks where the traction-free surface sits, exactly like `applyFreeSurface`/`material` in `HomogeneousElastic2DBenchmark_FreeSurface.ipynb`.

`topography-opt-*` and `topography-specfem` propagate OPT3 and SPECFEM2D on that model and are **not** SimuKirishima's own OPT3/FD3 cells — those run the OPT operator at a coarsened 200 m stride and use a simplified surface closure that does not match what `KirishimaElastic2DBenchmark.ipynb` documents. Instead they reuse the audited `clipped_available` weak-form closure from `HomogeneousElastic2DBenchmark_FreeSurface.ipynb` (`elastic2D_OPT3`/`elastic2D_OPT3_surface_clipped_available`, cache-compatible with that notebook since the recipe only depends on stencil geometry, not on this model's fields), adapted to `material2D`'s real, non-flat surface normals. FD3 is intentionally left out, matching this notebook's own stated scope (see the title cell).

Both the OPT3 propagation and the SPECFEM2D run are expensive on this 80 km × 42 km, 100 m grid and are off by default (`runKirishimaOPT3`, `runKirishimaSPECFEM2D` in `topography-opt-config`); set them to `true` to actually run. `topography-comparison` and `topography-video` degrade gracefully (a `@warn` and nothing else) until both runs exist.

In [ ]:
# Only the GeoPoints/planet1D module chain, needed to build the real
# Kirishima cross-section below. No Metal/backend and no flexOPT
# include here: those are only needed once the propagation part of
# this stage is wired up (see the note above).
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))
using .commonBatchs, .planet1D, .GeoPoints

In [ ]:
# Kirishima cross-section: real topography + NIED heterogeneous
# velocity model. Reuses only SimuKirishima.ipynb's construction
# primitives (constructLocalBox, getParamsAndTopo) at its own fine
# Δx=Δz=100 m — not its 3-D model (unneeded for a 2-D benchmark)
# and not its propagation cells.
kirishimaSummit = GeoPoint(dmsToDecimal(31, 56, 03), dmsToDecimal(130, 51, 42))
p0 = kirishimaSummit
Δx2D = 100.0 # m
Δz2D = 100.0 # m
altMax = 2.e3 # m
altMin = -40.e3 # m
horizontalHalfWidth = 40.e3 # m; the model spans 80 km horizontally

boxGrids2D = constructLocalBox(
    p0, Δx2D, Δz2D,
    -horizontalHalfWidth, horizontalHalfWidth,
    altMin, altMax,
)

niedVelocitySource = DEFAULT_NIED_VELOCITY_SOURCE[]
velocitySourceKey = nied_velocity_source_key(niedVelocitySource)
gridCacheTag2D = "$(join(size(boxGrids2D.allGridsInGeoPoints), 'x'))_" *
    "d$(boxGrids2D.Δx)_$(boxGrids2D.Δz)"
modelCacheName2D = "seismicModel2D_Kirishima_$(gridCacheTag2D)_NIED_$(velocitySourceKey)"
seismicModel2D = lazyProduceOrLoad(
    modelCacheName2D,
    getParamsAndTopo,
    boxGrids2D.allGridsInGeoPoints,
    boxGrids2D.effectiveRadii,
    0.1;
    velocity_model=:NIED,
    nied_source=niedVelocitySource,
    nied_confidence_max=0.8,
    nied_outside=:planet1D,
    nied_low_confidence=:planet1D,
)
@assert size(seismicModel2D.ρ) == size(boxGrids2D.allGridsInGeoPoints)

# getParamsAndTopo uses ρAir=0.001 by default; 0.01 cleanly separates
# rock from air/void (same threshold as SimuKirishima.ipynb).
air_density_cutoff = 0.01
material2D = seismicModel2D.ρ .> air_density_cutoff

xCoordinates2D = [p.xz[1] for p in boxGrids2D.allGridsInCartesian[:, 1]]
zCoordinates2D = [p.xz[2] for p in boxGrids2D.allGridsInCartesian[1, :]]
topographicSurfaceIndices2D = [
    findlast(@view material2D[ix, :]) for ix in axes(material2D, 1)]
@assert all(!isnothing, topographicSurfaceIndices2D) "a column is all air/void: widen altMax, or all solid: widen altMin"
topographicSurfaceIndices2D = Int.(topographicSurfaceIndices2D)
surfaceZ2D = [zCoordinates2D[k] for k in topographicSurfaceIndices2D]

@show size(material2D) count(material2D) extrema(surfaceZ2D)
@show seismicModel2D.velocity_model seismicModel2D.nied_source
@show extrema(seismicModel2D.Vpv[material2D]) extrema(seismicModel2D.Vsv[material2D])

topographyFigure = Figure(size=(900, 380))
topographyAxis = Axis(topographyFigure[1, 1];
    xlabel="x (km)", ylabel="z (km)", aspect=DataAspect(),
    title="Kirishima cross-section: topography-following material mask")
vsRange = extrema(seismicModel2D.Vsv[material2D])
heatmap!(topographyAxis, xCoordinates2D ./ 1e3, zCoordinates2D ./ 1e3,
    seismicModel2D.Vsv; colormap=:viridis, colorrange=vsRange)
lines!(topographyAxis, xCoordinates2D ./ 1e3, surfaceZ2D ./ 1e3;
    color=:white, linewidth=2, label="material/air interface")
Colorbar(topographyFigure[1, 2]; colormap=:viridis, colorrange=vsRange,
    label="Vs (km/s)")
axislegend(topographyAxis; position=:rb)
display(topographyFigure)

In [ ]:
# Metal/backend, flexOPT and specfemBenchmark — needed from here on for
# the OPT3/SPECFEM2D propagation. Mirrors the proven bootstrap in
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb, not SimuKirishima.ipynb's
# (which hard-errors when Metal is unavailable; detect_backend() already
# falls back to CPU on its own).
using Metal
include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
backend isa KernelAbstractions.CPU &&
    @warn("Running the OPT3 recipe construction on CPU; this will be slow")
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
include(joinpath(flexopt_root, "src", "specfemBenchmark.jl"))
using .flexOPT, .specfemBenchmark
set_default_form!(:weak)
@assert DEFAULT_FORM[] === :weak
@show backend

In [ ]:
runKirishimaOPT3 = true # expensive: set true explicitly
runKirishimaSPECFEM2D = true # expensive: set true explicitly

dxOPT = Δx2D
@assert Δx2D == Δz2D "the OPT3 recipe below assumes a square grid"
vpFieldOPT = seismicModel2D.Vpv .* 1e3   # m/s
vsFieldOPT = seismicModel2D.Vsv .* 1e3   # m/s
rhoFieldOPT = seismicModel2D.ρ .* 1e3    # kg/m^3
vpMaximum = maximum(vpFieldOPT[material2D])
# Conditioning constant only, exactly like rho0 in
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb: ρ, μ and λ are all
# divided by it, so it cancels out of the physical PDE it discretises.
rhoReference = median(rhoFieldOPT[material2D])
temporalRefinement = 2
dtOPT = 0.20 * dxOPT / (sqrt(2) * vpMaximum * temporalRefinement)

optFormulation = :scaled_coordinates
recipeSpacing = (1.0, 1.0, 1.0)
muOPT = (rhoFieldOPT .* vsFieldOPT.^2 ./ rhoReference) .* (dtOPT/dxOPT)^2
lambdaOPT = (rhoFieldOPT .* (vpFieldOPT.^2 .- 2vsFieldOPT.^2) ./ rhoReference) .*
    (dtOPT/dxOPT)^2
rhoOPT = rhoFieldOPT ./ rhoReference
muOPT[.!material2D] .= 0.0
lambdaOPT[.!material2D] .= 0.0

epicentreX2D = -10e3 # same epicentre as SimuKirishima.ipynb
sourceDepthBelowTopography2D = 2e3
sourceIX2D = argmin(abs.(xCoordinates2D .- epicentreX2D))
surfaceAltitudeAtSource2D = surfaceZ2D[sourceIX2D]
sourcePhysicalZ2D = surfaceAltitudeAtSource2D - sourceDepthBelowTopography2D
sourceIZ2D = argmin(abs.(zCoordinates2D .- sourcePhysicalZ2D))
@assert material2D[sourceIX2D, sourceIZ2D] "the source falls outside solid material; increase sourceDepthBelowTopography2D"

simulationDuration2D = 30.0 # s; includes several boundary reflections
outputSampling2D = 0.10 # s between stored/video frames
sourceFrequency2D = median(vsFieldOPT[material2D]) / (10dxOPT)
sourceDelay2D = min(12dtOPT, 0.25simulationDuration2D)
sourceForceAmplitude2D = 1.0e10 # N/m: line force in a unit-thickness slice
rickerSource2D(time) = begin
    a = π * sourceFrequency2D * (time - sourceDelay2D)
    (1 - 2a^2) * exp(-a^2)
end
# Soft check only, not a hard @assert: Vs dips locally near the surface of
# a volcanic edifice, and a single low-velocity cell should not block the
# whole run the way it correctly does in the homogeneous benchmark.
pointsPerSWavelength2D = minimum(vsFieldOPT[material2D]) / (sourceFrequency2D * dxOPT)

# Five receivers along the topographic surface, sampled two grid cells
# below the local elevation: the boundary row itself mixes surface/void
# degrees of freedom by construction and is not a clean displacement probe.
receiverOffsetsX2D = [-30e3, -20e3, 0.0, 20e3, 30e3]
receiverDepthBelowSurface2D = 2dxOPT
receiverGrid2D = map(receiverOffsetsX2D) do xr
    ix = argmin(abs.(xCoordinates2D .- xr))
    (x=xCoordinates2D[ix], z=surfaceZ2D[ix] - receiverDepthBelowSurface2D)
end
@assert all(receiverGrid2D) do r
    material2D[argmin(abs.(xCoordinates2D .- r.x)), argmin(abs.(zCoordinates2D .- r.z))]
end "a receiver falls outside solid material"

dtSPECFEM2D = dtOPT # exact common physical time step for OPT3 and SPECFEM2D
@show dxOPT dtOPT vpMaximum rhoReference
@show sourceFrequency2D sourceDelay2D pointsPerSWavelength2D
@show receiverGrid2D

In [ ]:
if runKirishimaOPT3
    optParameters2D = Dict{String,Any}(
        "famousEquationType" => "2DsismoTimeIsoHeteroSingleForce",
        "Δ" => recipeSpacing,
        "orderBtime" => 1, "orderBspace" => 1,
        "pointsInSpace" => 3, "pointsInTime" => 3,
        "supplementaryOrder" => 2,
        "variationalForm" => DEFAULT_FORM[],
        "taylor_inverse_mode" => :weak_operator_optimized,
        "fieldItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
            offsetTime=1, YorderBspace=-1, YorderBtime=-1),
        "materItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
            offsetTime=1, YorderBspace=-1, YorderBtime=-1),
        "recipe_backend" => backend,
    )
    # Same 3x2 clipped-available surface recipe as
    # HomogeneousElastic2DBenchmark_FreeSurface.ipynb, audited by
    # scripts/audit_opt3_weak_cross_partials.jl.
    clippedSurfaceParameters2D = copy(optParameters2D)
    clippedSurfaceParameters2D["pointsInSpace"] = (3, 2)
    clippedSurfaceParameters2D["nuCentre"] = (2, 2)
    clippedSurfaceParameters2D["fieldItpl"] = (
        ptsSpace=(3, 2), ptsTime=1, offsetSpace=(0.0, 0.0),
        offsetTime=1, YorderBspace=1, YorderBtime=-1,
    )
    clippedSurfaceParameters2D["trialFunctionRefPoints"] = (
        (1, 2, 3), (1, 2, 3), (1, 2, 3),
    )
    clippedSurfaceParameters2D["testIntegrationBounds"] = (
        (1.0, 3.0), (1.0, 2.0), (1.0, 3.0),
    )
    clippedSurfaceParameters2D["exactTaylorTotalDegree"] = 2

    # The weak-form basis only depends on the stencil geometry, not on
    # this model's fields, so this cache is shared with
    # HomogeneousElastic2DBenchmark_FreeSurface.ipynb's own recipes.
    recipeCacheVersion2D = 9
    function cachedOPTRecipe2D(parameters, prefix)
        cacheParameters = Dict{String,Any}(
            key => value for (key, value) in parameters if key != "recipe_backend")
        cacheParameters["recipe_cache_version"] = recipeCacheVersion2D
        function produceRecipe(config)
            runtimeParameters = Dict{String,Any}(config)
            pop!(runtimeParameters, "hash_id", nothing)
            pop!(runtimeParameters, "recipe_cache_version", nothing)
            runtimeParameters["recipe_backend"] = backend
            makeOPTsemiSymbolic(runtimeParameters)
        end
        myProduceOrLoad(produceRecipe, cacheParameters, "semiSymbolic", prefix)
    end
    optRecipe2D = cachedOPTRecipe2D(optParameters2D, "elastic2D_OPT3")
    clippedSurfaceRecipe2D = cachedOPTRecipe2D(
        clippedSurfaceParameters2D, "elastic2D_OPT3_surface_clipped_available")

    cerjan2D = CerjanBoundarySpec((24, 24), (24, 0); damping=0.0053)
    bcOPT2D = boundary_geometry(material2D, (dxOPT, dxOPT);
        free_surface_mode=:pinned_void, cerjan=cerjan2D)

    modelsOPT2D = [rhoOPT, lambdaOPT, muOPT]
    pointsOPT2D = getModelPoints(modelsOPT2D[1], 3,
        optRecipe2D["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT2D = (models=modelsOPT2D, modelPoints=pointsOPT2D,
        Δ=recipeSpacing, modelName="kirishima_topography_OPT3_$(optFormulation)")
    numericalVolume2D = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => optRecipe2D, "modelFam" => familyOPT2D,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPT2D, "representation" => "matrixfree",
    ))["numOperators"]
    preparedVolume2D = prepareLinearSystem(numericalVolume2D;
        free_surface_spacing=recipeSpacing[1:2])

    function prepare_surface_geometry2D(recipe, name)
        surfacePoints = getModelPoints(modelsOPT2D[1], 3,
            recipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
        surfaceFamily = (models=modelsOPT2D, modelPoints=surfacePoints,
            Δ=recipeSpacing, modelName=name)
        numerical = numericalOperatorConstruction(Dict{String,Any}(
            "optRec" => recipe, "modelFam" => surfaceFamily,
            "absorbingBoundaries" => nothing,
            "maskedRegionInSpace" => bcOPT2D.free_surface.points,
            "boundaryConditions" => nothing,
            "representation" => "matrixfree",
        ))["numOperators"]
        prepareLinearSystem(numerical)
    end
    preparedClippedSurface2D = prepare_surface_geometry2D(
        clippedSurfaceRecipe2D, "kirishima_topography_surface_clipped_available")
    surfaceWhole2D =
        numericalVolume2D.numericalOperators.left.geometry.freeSurfaceBoundary.points
    preparedOPT2D = overlapBoundaryLinearSystem(
        preparedVolume2D, preparedClippedSurface2D, surfaceWhole2D; mode=:replace)
    @assert preparedOPT2D.boundary_overlap_mode === :replace
    @assert !hasproperty(preparedOPT2D, :boundary_future_operator)

    A_surface2D = flexOPT.materializeConstantMatrix(preparedOPT2D)
    boundaryOverlapRows2D = hasproperty(preparedOPT2D, :boundary_overlap_rows) ?
        preparedOPT2D.boundary_overlap_rows : Int[]
    surfaceRowNorms2D = [norm(A_surface2D[row, :]) for row in boundaryOverlapRows2D]
    surfaceFactorization2D = try
        lu(A_surface2D)
        :regular
    catch err
        err isa SingularException ? :singular : rethrow()
    end
    @assert surfaceFactorization2D === :regular

    # Unlike the flat benchmark, the surface normals are not expected to
    # be vertical here: check that they are unit vectors pointing away
    # from the material instead, and report their actual range.
    surfaceNormals2D = bcOPT2D.free_surface.normals
    @assert all(n -> isapprox(hypot(n...), 1.0; atol=1e-6), surfaceNormals2D)
    @assert all(n -> n[2] > 0.0, surfaceNormals2D) "a surface normal points into the ground; check the material mask"
    freeSurfaceAudit2D = (
        surface_points=length(bcOPT2D.free_surface.points),
        overlap_rows=length(boundaryOverlapRows2D),
        zero_boundary_rows=count(iszero, surfaceRowNorms2D),
        normal_x_extrema=extrema(first.(surfaceNormals2D)),
        normal_z_extrema=extrema(last.(surfaceNormals2D)),
        surface_closure=:clipped_available,
    )
    display(freeSurfaceAudit2D)
else
    preparedOPT2D = nothing
    @warn "runKirishimaOPT3 = false; set it to true above and rerun to assemble OPT3"
end

In [ ]:
if runKirishimaOPT3
    optPadding2D = cerjan_padding(bcOPT2D.cerjan)
    xOPT2D = range(first(xCoordinates2D) - optPadding2D[1,1]*dxOPT;
        step=dxOPT, length=preparedOPT2D.spaceShape[1])
    zOPT2D = range(first(zCoordinates2D) - optPadding2D[1,2]*dxOPT;
        step=dxOPT, length=preparedOPT2D.spaceShape[2])
    sourceIndexOPT2D = CartesianIndex(
        sourceIX2D + optPadding2D[1,1], sourceIZ2D + optPadding2D[1,2])
    sourceLinearOPT2D = LinearIndices(preparedOPT2D.spaceShape)[sourceIndexOPT2D]

    optSteps2D = ceil(Int, simulationDuration2D / dtOPT)
    optOutputStride2D = max(1, round(Int, outputSampling2D / dtOPT))
    nPastForceLevels2D = preparedOPT2D.timePointsUsedForOneStep - 1
    sourceTimesOPT2D = ((1 - nPastForceLevels2D):optSteps2D) .* dtOPT
    waveletOPT2D = rickerSource2D.(sourceTimesOPT2D)
    # dt^2 F / (ρ Δx Δz): same force-to-displacement conversion as
    # elasticWave2D.add_ricker_source!(...; source_kind=:force).
    sourceDensityAtSource2D = rhoFieldOPT[sourceIX2D, sourceIZ2D]
    sourceScaleOPT2D = sourceForceAmplitude2D * dtOPT^2 /
        (sourceDensityAtSource2D * dxOPT^2)
    sourceOPT2D = zeros(Float64, preparedOPT2D.NforcePoints,
        preparedOPT2D.NForceField, length(sourceTimesOPT2D))
    sourceOPT2D[sourceLinearOPT2D, 2, :] .= sourceScaleOPT2D .* waveletOPT2D

    propagationOPT2D = propagateLinearSystem(
        preparedOPT2D, optSteps2D, dtOPT;
        sourceFull=sourceOPT2D, output_stride=optOutputStride2D,
        blowup_limit=1e8, solver_name="OPT3 clipped-available (Kirishima topography)",
        scheme=:direct,
    )
    @assert !propagationOPT2D.stopped_early
    uxOPT2D = propagationOPT2D.history[:, :, 1, :]
    uzOPT2D = propagationOPT2D.history[:, :, 2, :]
    optTimes2D = propagationOPT2D.times
    @show dtOPT size(uzOPT2D) maximum(abs, uxOPT2D) maximum(abs, uzOPT2D)
    @show propagationOPT2D.timing
else
    xOPT2D = zOPT2D = optTimes2D = uxOPT2D = uzOPT2D = nothing
    @warn "runKirishimaOPT3 = false; nothing to propagate"
end

In [ ]:
caseDirectory2D = joinpath(flexopt_root, "data", "specfem2d_benchmarks",
    "kirishima_topography_dx$(round(Int, dxOPT))m_nrec$(length(receiverGrid2D))")
specfemNxElements2D = round(Int,
    (last(xCoordinates2D)-first(xCoordinates2D)) / (4dxOPT))
specfemNzElements2D = round(Int,
    (last(zCoordinates2D)-first(zCoordinates2D)) / (4dxOPT))

# SPECFEM's tomography file is rectangular although the spectral mesh
# ends at the free surface: fill the unused samples above the topographic
# interface with the shallowest valid solid value (never sampled by the
# mesh). Same fix as KirishimaElastic2DBenchmark.ipynb's prepare-tomography
# cell; vpFieldOPT/vsFieldOPT/rhoFieldOPT (and material2D) are untouched,
# so OPT3 still sees the real void above the topography.
vpFieldSPECFEM2D = copy(vpFieldOPT)
vsFieldSPECFEM2D = copy(vsFieldOPT)
rhoFieldSPECFEM2D = copy(rhoFieldOPT)
for ix in axes(vsFieldSPECFEM2D, 1)
    valid = findall((@view vsFieldSPECFEM2D[ix, :]) .> 0)
    isempty(valid) && error("column $ix contains no elastic material")
    top = last(valid)
    for field in (vpFieldSPECFEM2D, vsFieldSPECFEM2D, rhoFieldSPECFEM2D)
        field[ix, top+1:end] .= field[ix, top]
    end
end
@assert minimum(vpFieldSPECFEM2D) > 0
@assert minimum(vsFieldSPECFEM2D) > 0
@assert minimum(rhoFieldSPECFEM2D) > 0

specfemCase2D = prepare_specfem2d_case(
    caseDirectory2D, xCoordinates2D, zCoordinates2D,
    vpFieldSPECFEM2D, vsFieldSPECFEM2D, rhoFieldSPECFEM2D, surfaceZ2D;
    source=(x=epicentreX2D, z=sourcePhysicalZ2D),
    receiver_points=receiverGrid2D,
    duration=simulationDuration2D, dt=dtSPECFEM2D, f0=sourceFrequency2D,
    source_factor=sourceForceAmplitude2D, source_angle=0.0,
    source_time_function=rickerSource2D,
    nx_elements=specfemNxElements2D, nz_elements=specfemNzElements2D,
    free_surface=true, record_at_surface_same_vertical=false,
    snapshot_interval_steps=max(1, round(Int, outputSampling2D / dtSPECFEM2D)),
    snapshot_image_type=5,
    output_wavefield_dumps=true, wavefield_dump_type=1, # displacement (uₓ, u_z)
    binary_wavefield_dumps=true,
)
specfemRun2D = if runKirishimaSPECFEM2D
    run_specfem2d_case(specfemCase2D.case_directory)
elseif isfile(joinpath(specfemCase2D.case_directory, "solver.log"))
    (output=specfemCase2D.output,)
else
    nothing
end

function integrate_velocity_trace2D(trace)
    # SPECFEM2D's seismotype=2 traces are velocity; OPT3's propagated
    # history is displacement (same conversion as
    # HomogeneousElastic2DBenchmark_FreeSurface.ipynb's specfem cell).
    displacement = zeros(Float64, length(trace.values))
    displacement[2:end] .= cumsum(
        ((trace.values[1:end-1] .+ trace.values[2:end]) ./ 2) .* diff(trace.time))
    (time=Float64.(trace.time), values=displacement)
end

if !isnothing(specfemRun2D)
    verticalFiles2D = find_specfem2d_traces(specfemRun2D.output;
        component=:z, network="FX")
    horizontalFiles2D = find_specfem2d_traces(specfemRun2D.output;
        component=:x, network="FX")
    @assert length(verticalFiles2D) == length(receiverGrid2D)
    @assert length(horizontalFiles2D) == length(receiverGrid2D)
    specfemTracesZ2D = [read_specfem2d_trace(file;
        time_shift=specfemCase2D.time_axis_shift) for file in verticalFiles2D]
    specfemTracesX2D = [read_specfem2d_trace(file;
        time_shift=specfemCase2D.time_axis_shift) for file in horizontalFiles2D]
    specfemWaveformsZ2D = integrate_velocity_trace2D.(specfemTracesZ2D)
    specfemWaveformsX2D = integrate_velocity_trace2D.(specfemTracesX2D)
    specfemWavefield2D = read_specfem2d_wavefield_dumps(specfemRun2D.output;
        dt=dtSPECFEM2D, time_axis_shift=specfemCase2D.time_axis_shift)
    @show specfemCase2D.case_directory dtSPECFEM2D
else
    specfemWaveformsZ2D = specfemWaveformsX2D = specfemWavefield2D = nothing
    @warn "runKirishimaSPECFEM2D = false; set it to true above and rerun to get a reference"
end

In [ ]:
if !isnothing(uxOPT2D) && !isnothing(specfemWaveformsZ2D)
    function sample_history2D(history, times, xaxis, zaxis, points)
        traces = Matrix{Float64}(undef, length(times), length(points))
        for (j, point) in enumerate(points)
            ix = argmin(abs.(xaxis .- point.x))
            iz = argmin(abs.(zaxis .- point.z))
            traces[:, j] .= Float64.(history[ix, iz, :])
        end
        (time=Float64.(times), values=traces)
    end
    optGridX2D = sample_history2D(uxOPT2D, optTimes2D, xOPT2D, zOPT2D, receiverGrid2D)
    optGridZ2D = sample_history2D(uzOPT2D, optTimes2D, xOPT2D, zOPT2D, receiverGrid2D)

    stationMetrics2D = map(eachindex(receiverGrid2D)) do station
        candidateZ = (time=optGridZ2D.time, values=optGridZ2D.values[:, station])
        candidateX = (time=optGridX2D.time, values=optGridX2D.values[:, station])
        (x_km=receiverGrid2D[station].x/1e3,
         uz=waveform_metrics(specfemWaveformsZ2D[station], candidateZ; samples=1001),
         ux=waveform_metrics(specfemWaveformsX2D[station], candidateX; samples=1001))
    end
    foreach(display, stationMetrics2D)

    waveformFitFigure2D = Figure(size=(1200, 210 * length(receiverGrid2D)))
    for (station, receiver) in enumerate(receiverGrid2D)
        for (column, (label, optGrid, specGrid)) in enumerate((
                ("x", optGridX2D, specfemWaveformsX2D),
                ("z", optGridZ2D, specfemWaveformsZ2D)))
            axis = Axis(waveformFitFigure2D[station, column];
                xlabel=station == length(receiverGrid2D) ? "time (s)" : "",
                ylabel="x=$(round(receiver.x/1e3; digits=1)) km",
                title=station == 1 ? "u$label" : "")
            lines!(axis, optGrid.time, optGrid.values[:, station];
                label="OPT3", color=:darkorange)
            lines!(axis, specGrid[station].time, specGrid[station].values;
                label="SPECFEM2D", color=:black, linewidth=2)
            station == 1 && column == 1 && axislegend(axis; position=:rt, labelsize=9)
        end
    end
    display(waveformFitFigure2D)
else
    @warn "OPT3 and/or SPECFEM2D results are missing; set the run flags in topography-opt-config to true"
end
nothing

In [ ]:
if !isnothing(uxOPT2D) && !isnothing(specfemWavefield2D)
    nearest_frame2D(times, time) = argmin(abs.(times .- time))
    function record_kirishima_comparison(filepath; video_dt=0.1, framerate=15)
        frameTimes = collect(max(first(optTimes2D), first(specfemWavefield2D.time)):
            video_dt:min(last(optTimes2D), last(specfemWavefield2D.time)))
        commonPeak = max(maximum(abs, uzOPT2D), maximum(abs, specfemWavefield2D.uz),
            eps(Float64))
        colorRange = (-commonPeak, commonPeak)
        currentTime = Observable(first(frameTimes))
        optFrame = Observable(Float32.(uzOPT2D[:, :, 1]))
        specfemFrame = Observable(Float32.(specfemWavefield2D.uz[:, :, 1]))
        figure = Figure(size=(1300, 560))
        optAxis = Axis(figure[1, 1]; xlabel="x (km)", ylabel="z (km)",
            title=@lift("OPT3 — t = $(round($currentTime; digits=2)) s"),
            aspect=DataAspect())
        specfemAxis = Axis(figure[1, 2]; xlabel="x (km)", ylabel="z (km)",
            title=@lift("SPECFEM2D — t = $(round($currentTime; digits=2)) s"),
            aspect=DataAspect())
        Label(figure[0, 1:2], "Kirishima topography, vertical displacement u_z"; fontsize=20)
        optPlot = heatmap!(optAxis, collect(xOPT2D)./1e3, collect(zOPT2D)./1e3,
            optFrame; colormap=:balance, colorrange=colorRange)
        heatmap!(specfemAxis, specfemWavefield2D.x./1e3, specfemWavefield2D.z./1e3,
            specfemFrame; colormap=:balance, colorrange=colorRange)
        lines!(optAxis, xCoordinates2D./1e3, surfaceZ2D./1e3; color=:black, linewidth=1)
        lines!(specfemAxis, xCoordinates2D./1e3, surfaceZ2D./1e3; color=:black, linewidth=1)
        for axis in (optAxis, specfemAxis)
            scatter!(axis, [epicentreX2D/1e3], [sourcePhysicalZ2D/1e3];
                marker=:star5, color=:gold, strokecolor=:black, markersize=16)
        end
        Colorbar(figure[1, 3], optPlot; label="u_z (m)")
        record(figure, filepath, frameTimes; framerate=framerate) do time
            currentTime[] = time
            optFrame[] = Float32.(uzOPT2D[:, :, nearest_frame2D(optTimes2D, time)])
            specfemFrame[] = Float32.(specfemWavefield2D.uz[:, :,
                nearest_frame2D(specfemWavefield2D.time, time)])
        end
        filepath
    end
    videoDirectory2D = joinpath(flexopt_root, "data", "kirishimaElastic2DTopography")
    mkpath(videoDirectory2D)
    kirishimaVideo2D = record_kirishima_comparison(
        joinpath(videoDirectory2D, "OPT3_SPECFEM2D_topography.mp4"))
    @show kirishimaVideo2D
else
    @warn "OPT3 and/or SPECFEM2D wavefields are missing; set the run flags in topography-opt-config to true"
    nothing
end